# 02. Spark & Delta Lake Integration Test

Notebook này dùng để kiểm tra cấu hình Spark, khả năng đọc ghi vào Delta Lake trên MinIO (S3) và tính năng Time Travel.

In [1]:
import sys
import os
from pyspark.sql.types import StructType, StructField, LongType, FloatType

# Thêm thư mục gốc vào path để import src
sys.path.append(os.path.abspath('..'))

from src.utils.spark_session import get_spark_session
from pyspark.sql import functions as F

## 1. Khởi tạo Spark Session

In [2]:
spark = get_spark_session("Notebook-Infrastructure-Test")
print(f"Spark Version: {spark.version}")

Spark Version: 3.4.1


## 2. Kiểm tra đọc dữ liệu Local và ghi vào Delta Lake (MinIO)

In [3]:
# 1. Khai báo Schema thủ công để Spark chạy siêu tốc (< 2s)
manual_schema = StructType([
    StructField("userId", LongType(), True),
    StructField("movieId", LongType(), True),
    StructField("rating", FloatType(), True),
    StructField("timestamp", LongType(), True)
])

# 2. Đọc thử 1000 dòng ratings
df = spark.read.csv("../dataset/ml-25m/ratings.csv", header=True, schema=manual_schema).limit(1000)

# 3. Ghi vào Delta Lake trên MinIO
TEST_PATH = "s3a://movie-data/test/delta_test_table"
df.write.format("delta").mode("overwrite").save(TEST_PATH)

print("Ghi dữ liệu vào Delta Lake thành công!")

Ghi dữ liệu vào Delta Lake thành công!


## 3. Kiểm tra tính năng Time Travel

In [4]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, TEST_PATH)

# Xem lịch sử phiên bản
delta_table.history().select("version", "timestamp", "operation").show()

+-------+-------------------+---------+
|version|          timestamp|operation|
+-------+-------------------+---------+
|      2|2026-04-22 00:31:17|    WRITE|
|      1|2026-04-22 00:23:30|    WRITE|
|      0|2026-04-22 00:22:16|    WRITE|
+-------+-------------------+---------+



## 4. Dọn dẹp

In [ ]:
spark.stop()